In [2]:
import os
import shutil

# 1. Create the wavs folder if it doesn't exist
audio_dir = "/content/wavs"
if not os.path.exists(audio_dir):
    os.makedirs(audio_dir)

# 2. Move all individual .wav files from /content/ to /content/wavs/
files_moved = 0
for file in os.listdir("/content/"):
    if file.endswith(".wav"):
        shutil.move(os.path.join("/content/", file), os.path.join(audio_dir, file))
        files_moved += 1

print(f"✅ Successfully moved {files_moved} files into {audio_dir}")

✅ Successfully moved 0 files into /content/wavs


In [3]:
# 1. Clone and install Parler-TTS with training dependencies
!git clone https://github.com/huggingface/parler-tts.git
%cd parler-tts
!pip install -e .[train]
!pip install --upgrade protobuf wandb==0.16.6

fatal: destination path 'parler-tts' already exists and is not an empty directory.
/content/parler-tts
Obtaining file:///content/parler-tts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-1eaopra9/descript-audiotools_e1e8ec52baae44ecb34816b960688a9a
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-1eaopra9/descript-audiotools_e1e8ec52baae44ecb34816b960688a9a
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Preparing metadata (setup.py) ... done
  Building editable for parler_tts (pyproject.toml) ... done
  Created wheel for parler_tts: filename=parler_tts-0.2.2-0.editable-py3-none-any.whl size=10917 sha256=ee635685df406426ada377f2411b9ba2

In [4]:
import pandas as pd
import os

# 1. Load the TSV without headers (header=None)
# Based on your output:
# Column 1 (index 1) = Audio filename
# Column 3 (index 3) = Punjabi Text
df = pd.read_csv('/content/test.tsv', sep='\t', header=None)

# 2. Map the data to a clean structure
audio_folder = '/content/wavs/'

# Create a new, clean DataFrame
clean_df = pd.DataFrame()

# index 1 is the .wav filename column
clean_df['audio'] = df[1].apply(lambda x: os.path.join(audio_folder, str(x)))

# index 3 is the clean Punjabi text column (without the bars | )
clean_df['text'] = df[3]

# 3. Add the description (required for Parler-TTS)
# Note: Your data shows both MALE and FEMALE speakers.
# We'll use a neutral description so the model learns the voice correctly.
clean_df['description'] = "A speaker speaks clearly in English with a natural tone and moderate pace in a quiet environment."

# 4. Save to the final CSV
clean_df.to_csv('/content/English_perfect.csv', index=False)

print("✅ Successfully fixed! Your new dataset has:")
print(f"- Total rows: {len(clean_df)}")
print(clean_df.head(2))

✅ Successfully fixed! Your new dataset has:
- Total rows: 10
                                    audio  \
0  /content/wavs/12741024238657315067.wav   
1   /content/wavs/9055240296608129734.wav   

                                                text  \
0  unfortunately studying traffic flow is difficu...   
1  unfortunately studying traffic flow is difficu...   

                                         description  
0  A speaker speaks clearly in English with a nat...  
1  A speaker speaks clearly in English with a nat...  


In [5]:
!git clone https://github.com/huggingface/parler-tts.git
%cd parler-tts


fatal: destination path 'parler-tts' already exists and is not an empty directory.
/content/parler-tts/parler-tts


In [6]:
!pip install -e .[train]
!pip install datasets accelerate wandb


Obtaining file:///content/parler-tts/parler-tts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-0ztj5nqq/descript-audiotools_060f212ab7aa4000b4abdcafe8ca3925
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-0ztj5nqq/descript-audiotools_060f212ab7aa4000b4abdcafe8ca3925
  Resolved https://github.com/descriptinc/audiotools to commit 348ebf2034ce24e2a91a553e3171cb00c0c71678
  Preparing metadata (setup.py) ... done
  Building editable for parler_tts (pyproject.toml) ... done
  Created wheel for parler_tts: filename=parler_tts-0.2.2-0.editable-py3-none-any.whl size=10919 sha256=a35b52b509972a1ce91d327140bb2ec592de5f813b0cbab9173b7c1eab456045
  Stored in directory: /tmp/pip-ephem-wheel-cache-bdc1vap0/

In [7]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/content/English_perfect.csv"
)

dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['audio', 'text', 'description'],
        num_rows: 10
    })
})

In [8]:
from datasets import Audio

dataset = dataset.cast_column("audio", Audio(sampling_rate=44100))


In [9]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "parler-tts/parler-tts-mini-v1"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [10]:
def prepare(example):
    example["input_ids"] = tokenizer(example["text"]).input_ids
    example["description_ids"] = tokenizer(example["description"]).input_ids
    return example

dataset = dataset.map(prepare)


Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [11]:
!pip install numpy==1.26.4


In [12]:
!git clone https://github.com/huggingface/parler-tts.git
%cd parler-tts
!pip install -e .[train]
!pip install datasets accelerate wandb


Cloning into 'parler-tts'...
remote: Enumerating objects: 1100, done.
remote: Counting objects: 100% (458/458), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 1100 (delta 345), reused 280 (delta 280), pack-reused 642 (from 2)
Receiving objects: 100% (1100/1100), 354.82 KiB | 5.00 MiB/s, done.
Resolving deltas: 100% (703/703), done.
/content/parler-tts/parler-tts/parler-tts
Obtaining file:///content/parler-tts/parler-tts/parler-tts
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Cloning https://github.com/descriptinc/audiotools to /tmp/pip-install-h_i2pm6k/descript-audiotools_16b3226f16e8456683385b11a797a280
  Running command git clone --filter=blob:none --quiet https://github.com/descriptinc/audiotools /tmp/pip-install-h_i2pm6k/descript-audiotools_16b3226f16e8456683385b11a797a280
  Resolved https://gith

In [13]:
!accelerate config default


Configuration already exists at /root/.cache/huggingface/accelerate/default_config.yaml, will not override. Run `accelerate config` manually or pass a different `save_location`.


In [ ]:
!python training/run_parler_tts_training.py --help


2026-03-06 07:13:02.191407: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772781182.212842    7299 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772781182.220108    7299 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772781182.238375    7299 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772781182.238401    7299 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772781182.238405    7299 computation_placer.cc:177] computation placer alr

In [14]:
!accelerate launch training/run_parler_tts_training.py \
--model_name_or_path "parler-tts/parler-tts-mini-v1" \
--train_dataset_name "csv" \
--data_files "/content/English_perfect.csv" \
--target_audio_column_name "audio" \
--prompt_column_name "text" \
--description_column_name "description" \
--do_train true \
--num_train_epochs 5 \
--per_device_train_batch_size 2 \
--gradient_accumulation_steps 4 \
--learning_rate 5e-5 \
--logging_steps 10 \
--save_steps 100 \
--dtype "float16" \
--output_dir "/content/parler_finetuned"


2026-03-06 07:50:38.700448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772783438.724411   27426 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772783438.732326   27426 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772783438.750608   27426 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772783438.750635   27426 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772783438.750639   27426 computation_placer.cc:177] computation placer alr

In [9]:
!accelerate launch ./training/run_parler_tts_training.py \
--model_name_or_path "./parler-tts-untrained-600M/parler-tts-untrained-600M/" \
--feature_extractor_name "parler-tts/dac_44khZ_8kbps" \
--description_tokenizer_name "google/flan-t5-large" \
--prompt_tokenizer_name "google/flan-t5-large" \
--report_to "none" \
--overwrite_output_dir true \
--train_dataset_name "parler-tts/libritts_r_filtered+parler-tts/libritts_r_filtered+parler-tts/libritts_r_filtered+parler-tts/mls_eng" \
--train_metadata_dataset_name "parler-tts/libritts-r-filtered-speaker-descriptions+parler-tts/libritts-r-filtered-speaker-descriptions+parler-tts/libritts-r-filtered-speaker-descriptions+parler-tts/mls-eng-speaker-descriptions" \
--train_dataset_config_name "clean+clean+other+default" \
--train_split_name "train.clean.360+train.clean.100+train.other.500+train" \
--eval_dataset_name "parler-tts/libritts_r_filtered+parler-tts/mls_eng" \
--eval_metadata_dataset_name "parler-tts/libritts-r-filtered-speaker-descriptions+parler-tts/mls-eng-speaker-descriptions" \
--eval_dataset_config_name "other+default" \
--eval_split_name "test.other+test" \
--target_audio_column_name "audio" \
--description_column_name "text_description" \
--prompt_column_name "text" \
--max_duration_in_seconds 30 \
--min_duration_in_seconds 2.0 \
--max_text_length 600 \
--add_audio_samples_to_wandb true \
--id_column_name "id" \
--preprocessing_num_workers 8 \
--do_train true \
--num_train_epochs 4 \
--gradient_accumulation_steps 6 \
--gradient_checkpointing false \
--per_device_train_batch_size 4 \
--learning_rate 0.00095 \
--adam_beta1 0.9 \
--adam_beta2 0.99 \
--weight_decay 0.01 \
--lr_scheduler_type "constant_with_warmup" \
--warmup_steps 20000 \
--logging_steps 1000 \
--freeze_text_encoder true \
--do_eval true \
--predict_with_generate true \
--include_inputs_for_metrics true \
--evaluation_strategy steps \
--eval_steps 10000 \
--save_steps 10000 \
--per_device_eval_batch_size 4 \
--audio_encoder_per_device_batch_size 24 \
--dtype "float16" \
--seed 456 \
--output_dir "./output_dir_training/" \
--temporary_save_to_disk "./audio_code_tmp/" \
--save_to_disk "./tmp_dataset_audio/" \
--max_eval_samples 96 \
--dataloader_num_workers 8 \
--group_by_length true \
--attn_implementation "sdpa"

Streaming output truncated to the last 5000 lines.
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064.par(…):  23% 134M/577M [00:22<00:00, 655MB/s]
clean/train.clean.360-00032-of-00064

KeyboardInterrupt: 